In [ ]:
import pandas as pd
from IPython.display import display
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import MinMaxScaler
import torch
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Load CSV and split features and labels
df = pd.read_csv('null-corrected.csv', index_col=0)
display(df.head())
X = df.iloc[:, :-1]
# Convert Y to integer labels for classification
Y = df.iloc[:, -1].astype(int)
display(X.head())
display(Y.head(n=10))

# Preprocess features: normalize to [-1, 1]
scaler_X = MinMaxScaler(feature_range=(-1, 1))
X_scaled = scaler_X.fit_transform(X)

# For classification, we keep Y as integer labels (0, 1, 2, or 3)
Y_tensor = torch.tensor(Y.values, dtype=torch.long)

# Set device to CUDA if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# LOOCV setup
loo = LeaveOneOut()
folds = list(loo.split(X_scaled))
input_dim = X_scaled.shape[1]
num_classes = 5

total_loss = 0.0
correct_count = 0  # for accuracy
count = 0

loss_history = []

# Outer loop for LOOCV folds
for fold_idx, (train_index, test_index) in enumerate(tqdm(folds, desc="LOOCV Folds"), start=1):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    Y_train, Y_test = Y_tensor[train_index], Y_tensor[test_index]
    
    # Convert training data to torch tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    Y_train_tensor = Y_train.clone().detach()
    
    # Convert test data and move to device
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    Y_test_tensor = Y_test.clone().detach().to(device)
    
    # Create DataLoader for training data
    train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                              pin_memory=True if device.type == 'cuda' else False)
    
    # Define logistic regression model (no activation needed as CrossEntropyLoss applies softmax)
    model = torch.nn.Sequential(
        torch.nn.Linear(input_dim, num_classes),
        # torch.nn.Sigmoid()
    ).to(device)
    
    # Use CrossEntropyLoss for multi-class classification
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    
    fold_loss_history = []
    epochs = 10
    # Inner loop for epochs
    for epoch in tqdm(range(epochs), desc=f"Fold {fold_idx} Epochs", leave=False):
        epoch_loss = 0.0
        for inputs, labels in train_loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item() * inputs.size(0)
        avg_loss = epoch_loss / len(train_loader.dataset)
        fold_loss_history.append(avg_loss)
    loss_history.append(fold_loss_history)
    
    # Evaluate on the left-out sample
    with torch.no_grad():
        output = model(X_test_tensor)
        loss = criterion(output, Y_test_tensor)
        total_loss += loss.item()
        
        # Get the predicted class (largest logit)
        pred_class = output.argmax(dim=1, keepdim=True)
        if pred_class.item() == Y_test_tensor.item():
            correct_count += 1
        
        count += 1

print(f'\nLOOCV Mean Loss: {total_loss/count:.4f}')
accuracy = correct_count / count * 100
print(f'LOOCV Accuracy: {accuracy:.2f}%')
